<a href="https://colab.research.google.com/github/Sai875/flyrank-ml-internship/blob/main/work/notebooks/Google_Search_Ranking_and_Discoverability_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — Search Traffic Decay & Content Action Playbook

This notebook evaluates whether a compact Logistic Regression model can rank records associated with an observed decline proxy better than a transparent hand-built heuristic. The unit of analysis is one pseudonymized content record from the 30,000-row starter CSV. The validation split holds out entire client groups. The CI workflow reruns this notebook from a fresh kernel and checks aggregate metrics, the leakage control, and the report-safe figure.

**Interpretation boundary:** this is retrospective prioritization for human review, not future forecasting, a causal analysis, or a guarantee that content changes recover traffic.

**Repository note:** The automated GitHub Actions run uses `work/notebooks/capstone.ipynb` as its canonical source. This descriptive filename mirrors that analysis for easier discovery; keep the two notebooks aligned when making edits.

In [1]:
import sys
import os
import json
import shutil
import subprocess
from pathlib import Path
from datetime import datetime, timezone

%pip -q install numpy pandas matplotlib scikit-learn huggingface_hub ipywidgets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
REPO_URL = "https://github.com/Sai875/flyrank-ml-internship.git"
REPO_DIR = Path("/content/flyrank-ml-internship")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    # GitHub Actions starts in the repository root; local Jupyter may start elsewhere.
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "work/notebooks/capstone.ipynb").exists():
            os.chdir(candidate)
            break

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
if not DATA_PATH.exists():
    from huggingface_hub import hf_hub_download
    downloaded = hf_hub_download(
        repo_id="FlyRank/internship-starter",
        repo_type="dataset",
        filename="content_refresh_anonymized.csv",
    )
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(downloaded, DATA_PATH)

df = pd.read_csv(DATA_PATH)
required = {
    "trend_direction", "trend_pct", "client_id", "avg_position", "days_since_last_update",
    "impressions_90d", "ctr", "content_type",
}
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"Required columns missing from the starter dataset: {missing}")
if df.empty:
    raise ValueError("The starter dataset is empty.")

TARGET = "is_declining_label"
df[TARGET] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
for column in ("days_since_last_update", "impressions_90d", "ctr", "avg_position"):
    df[column] = pd.to_numeric(df[column], errors="coerce")
df["trend_pct"] = pd.to_numeric(df["trend_pct"], errors="coerce")
df["avg_position_no_data"] = df["avg_position"].fillna(0).eq(0).astype(int)
df["avg_position_model"] = df["avg_position"].replace(0, np.nan)
df["content_type"] = df["content_type"].fillna("unknown").astype(str)

NUMERIC = [
    "days_since_last_update", "impressions_90d", "ctr",
    "avg_position_model", "avg_position_no_data",
]
CATEGORICAL = ["content_type"]
FEATURES = NUMERIC + CATEGORICAL
FORBIDDEN = {"trend_direction", "trend_pct", TARGET, "content_id", "client_id"}
assert not (set(FEATURES) & FORBIDDEN), f"Forbidden feature(s): {set(FEATURES) & FORBIDDEN}"

print(f"Loaded {len(df):,} rows × {df.shape[1]} columns across {df['client_id'].nunique():,} client groups.")
print(f"Observed decline-proxy rate: {df[TARGET].mean():.3f}")


Note: you may need to restart the kernel to use updated packages.


content_refresh_anonymized.csv:   0%|          | 0.00/8.23M [00:00<?, ?B/s]

Loaded 30,000 rows × 56 columns across 32 client groups.
Observed decline-proxy rate: 0.542


## 1. Method and validation

The target is `trend_direction == "down"`, an observed proxy from the starter snapshot. The model uses five numeric features and one categorical feature. Identifiers and label-source fields are excluded. Zero average position is treated as missing and represented separately.

The holdout is a deterministic 20% split by `client_id`, so records from one client cannot appear in both train and test. A transparent heuristic baseline ranks pages by a weighted combination of visibility, freshness risk, position opportunity, and content-depth gap. The formula is a triage comparator, not a trained model.

In [2]:
def percentile_rank(series):
    return pd.to_numeric(series, errors="coerce").fillna(0).rank(method="average", pct=True).fillna(0)

def normalize(series):
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    lo, hi = values.min(), values.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - lo) / (hi - lo)

# Transparent rule baseline: uses observed operational signals, not label-derived fields.
impressions = pd.to_numeric(df["impressions_90d"], errors="coerce").fillna(0)
age = pd.to_numeric(df["days_since_last_update"], errors="coerce").fillna(0)
position = pd.to_numeric(df["avg_position"], errors="coerce").fillna(0)
word_count = pd.to_numeric(df["word_count"], errors="coerce").fillna(0) if "word_count" in df.columns else pd.Series(np.zeros(len(df)), index=df.index)

visibility_score = percentile_rank(np.log1p(impressions))
freshness_risk_score = percentile_rank(age)
position_opportunity_score = (
    normalize(position.clip(lower=1, upper=50))
    * visibility_score
    * (position > 0).astype(int)
)
depth_gap_score = (1 - percentile_rank(word_count)) * visibility_score
df["baseline_score"] = (
    0.40 * visibility_score
    + 0.30 * freshness_risk_score
    + 0.25 * position_opportunity_score
    + 0.05 * depth_gap_score
).clip(0, 1)

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
categorical_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, NUMERIC),
    ("cat", categorical_pipe, CATEGORICAL),
])
model = Pipeline([
    ("preprocess", preprocess),
    ("logreg", LogisticRegression(max_iter=1000, random_state=SEED)),
])

splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, test_idx = next(splitter.split(df, y=df[TARGET], groups=df["client_id"]))
train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()
train_clients = set(train_df["client_id"].astype(str))
test_clients = set(test_df["client_id"].astype(str))
overlap = train_clients & test_clients
assert not overlap, f"Client overlap detected: {len(overlap)}"
model.fit(train_df[FEATURES], train_df[TARGET])
model_scores = model.predict_proba(test_df[FEATURES])[:, 1]
baseline_scores = test_df["baseline_score"].to_numpy()

def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    order = np.argsort(-scores, kind="mergesort")[:min(k, len(scores))]
    return float(y_true[order].mean()) if len(order) else 0.0

ks = [10, 20, 50, 100]
test_y = test_df[TARGET].astype(int).to_numpy()
test_base_rate = float(test_y.mean())
precision_rows = []
for k in ks:
    precision_rows.append({
        "k": k,
        "metric": f"Precision@{k}",
        "Week-4 baseline": precision_at_k(test_y, baseline_scores, k),
        "Logistic Regression": precision_at_k(test_y, model_scores, k),
        "test label rate": test_base_rate,
    })

# Deliberate leakage negative control: trend_pct is a source of the proxy label,
# so it is intentionally included only in this separate invalid model.
LEAKAGE_FEATURES = NUMERIC + ["trend_pct"] + CATEGORICAL
leakage_preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), NUMERIC + ["trend_pct"]),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), CATEGORICAL),
])
leakage_model = Pipeline([
    ("preprocess", leakage_preprocess),
    ("logreg", LogisticRegression(max_iter=1000, random_state=SEED)),
])
leakage_model.fit(train_df[LEAKAGE_FEATURES], train_df[TARGET])
leakage_scores = leakage_model.predict_proba(test_df[LEAKAGE_FEATURES])[:, 1]
leakage_precision50 = precision_at_k(test_y, leakage_scores, 50)
leakage_roc_auc = float(roc_auc_score(test_y, leakage_scores))
leakage_table = pd.DataFrame([
    {"experiment": "Honest final feature set", "Precision@50": precision_at_k(test_y, model_scores, 50), "ROC-AUC": float(roc_auc_score(test_y, model_scores))},
    {"experiment": "INVALID: trend_pct added", "Precision@50": leakage_precision50, "ROC-AUC": leakage_roc_auc},
])
display(leakage_table.round(3))
assert "trend_pct" not in FEATURES
assert "trend_pct" not in NUMERIC

results_df = pd.DataFrame(precision_rows)
display(results_df.round(3))


,experiment,Precision@50,ROC-AUC
0,Honest final feature set,0.62,0.563
1,INVALID: trend_pct added,1.00,0.955


,k,metric,Week-4 baseline,Logistic Regression,test label rate
0,10,Precision@10,0.10,0.50,0.511
1,20,Precision@20,0.30,0.55,0.511
2,50,Precision@50,0.32,0.62,0.511
3,100,Precision@100,0.35,0.60,0.511


## 2. Results and limits

The table above is recomputed from the current downloaded dataset and deterministic grouped split. A separate, deliberately invalid negative control adds `trend_pct`, a label-source field, to show how leakage can inflate evaluation; that feature is excluded from the final feature list. Use the displayed values and saved JSON as the source of truth; do not hard-code old metric values into conclusions.

The decline proxy is not a verified future outcome. This snapshot does not establish causality, predict Google's ranking algorithm, or show that a refresh will recover performance. A high-ranked page is a candidate for human review only.

In [3]:
baseline_roc_auc = float(roc_auc_score(test_y, baseline_scores))
model_roc_auc = float(roc_auc_score(test_y, model_scores))
baseline_ap = float(average_precision_score(test_y, baseline_scores))
model_ap = float(average_precision_score(test_y, model_scores))

metrics_payload = {
    "experiment": "capstone_client_grouped_holdout",
    "seed": SEED,
    "split": "GroupShuffleSplit test_size=0.20 grouped by client_id",
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "train_clients": int(len(train_clients)),
    "test_clients": int(len(test_clients)),
    "overlapping_clients": int(len(overlap)),
    "test_label_rate": test_base_rate,
    "features_numeric": NUMERIC,
    "features_categorical": CATEGORICAL,
    "metrics": [
        {key: (int(value) if key == "k" else float(value) if key in ("Week-4 baseline", "Logistic Regression", "test label rate") else value)
         for key, value in row.items()}
        for row in precision_rows
    ],
    "baseline_roc_auc": baseline_roc_auc,
    "model_roc_auc": model_roc_auc,
    "baseline_average_precision": baseline_ap,
    "model_average_precision": model_ap,
    "leakage_control": {
        "feature": "trend_pct",
        "included_in_final_features": False,
        "precision_at_50": leakage_precision50,
        "roc_auc": leakage_roc_auc,
        "interpretation": "Invalid negative control only; never use this feature for final predictions.",
    },
    "caveat": "Observed decline proxy; decision support only; no causal claims.",
    "generated_utc": datetime.now(timezone.utc).isoformat(),
}

# Explicit track-rubric deliverables: 3–5 evidence-backed insights and a presentation outline.
row_by_k = {int(row["k"]): row for row in precision_rows}
p50_model = float(row_by_k[50]["Logistic Regression"])
p50_baseline = float(row_by_k[50]["Week-4 baseline"])
p50_lift = p50_model / test_base_rate if test_base_rate > 0 else None
evidence_backed_insights = [
    {
        "id": "I1",
        "finding": "The learned ranking beats the transparent rule at each evaluated top-K cutoff.",
        "evidence": "; ".join(
            f"Precision@{k}: model {float(row_by_k[k]['Logistic Regression']):.3f} vs rule {float(row_by_k[k]['Week-4 baseline']):.3f}"
            for k in (10, 20, 50, 100)
        ),
        "action": "Keep the model as a candidate ordering aid for a human-reviewed queue; compare it against the rule on each new evaluation.",
    },
    {
        "id": "I2",
        "finding": "The strongest claim is top-K triage, not strong general-purpose prediction.",
        "evidence": (
            f"Precision@50 {p50_model:.3f} vs rule {p50_baseline:.3f}; "
            f"test proxy rate {test_base_rate:.3f}; lift {p50_lift:.2f}x; "
            f"model ROC-AUC {model_roc_auc:.3f}; average precision {model_ap:.3f}."
        ),
        "action": "Choose K from actual reviewer capacity and validate on a fresh time window; never interpret ranking scores as calibrated recovery probabilities.",
    },
    {
        "id": "I3",
        "finding": "Client grouping avoids client leakage but evidence for new-client generalization is still limited.",
        "evidence": (
            f"Train: {len(train_df):,} rows / {len(train_clients)} clients; "
            f"test: {len(test_df):,} rows / {len(test_clients)} clients; "
            f"overlapping clients: {len(overlap)}."
        ),
        "action": "Repeat evaluation across multiple client-grouped seeds and a future time window before operational adoption.",
    },
    {
        "id": "I4",
        "finding": "A label-source field can create deceptively excellent scores and must stay excluded.",
        "evidence": (
            f"Adding invalid trend_pct produces Precision@50 {leakage_precision50:.3f} and ROC-AUC {leakage_roc_auc:.3f}; "
            f"the honest model has Precision@50 {p50_model:.3f} and ROC-AUC {model_roc_auc:.3f}."
        ),
        "action": "Retain the feature-exclusion assertion and rerun leakage checks whenever the target or feature frame changes.",
    },
]
presentation_outline = [
    {"slide": 1, "title": "Problem and decision", "key_points": "Too many pages for manual review; rank the first pages to inspect, not the pages guaranteed to recover."},
    {"slide": 2, "title": "Data and target", "key_points": "30,000-row anonymized starter snapshot; target is trend_direction == down; public-safe aggregates only."},
    {"slide": 3, "title": "Method and validation", "key_points": "Logistic Regression vs transparent rule; grouped by client; seed 42; zero client overlap."},
    {"slide": 4, "title": "Results at reviewer capacity", "key_points": "Precision@10/20/50/100; compare model, baseline and test proxy rate; emphasize Precision@50."},
    {"slide": 5, "title": "Leakage and limitations", "key_points": "trend_pct negative control inflates results; modest ROC-AUC; seven held-out clients; snapshot label is not future outcome."},
    {"slide": 6, "title": "Action playbook and next experiment", "key_points": "Human-check measurement, intent, CTR, visibility and freshness cues; test on a future window; no automatic edits."},
]
metrics_payload["evidence_backed_insights"] = evidence_backed_insights
metrics_payload["presentation_outline"] = presentation_outline

OUTPUT_DIR = Path("work/outputs")
FIGURE_DIR = Path("work/figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = OUTPUT_DIR / "capstone_metrics.json"
FIGURE_PATH = FIGURE_DIR / "capstone_precision_at_k.png"
METRICS_PATH.write_text(json.dumps(metrics_payload, indent=2) + "\n", encoding="utf-8")

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(ks))
width = 0.24
ax.bar(x - width, [r["Week-4 baseline"] for r in precision_rows], width, label="Week-4 baseline")
ax.bar(x, [r["Logistic Regression"] for r in precision_rows], width, label="Logistic Regression")
ax.bar(x + width, [r["test label rate"] for r in precision_rows], width, label="Test label rate")
ax.set_xticks(x, [f"@{k}" for k in ks])
ax.set_ylim(0, 1)
ax.set_ylabel("Precision")
ax.set_title("Precision@K on client-grouped holdout")
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_PATH, dpi=160, bbox_inches="tight")
plt.show()

display(Markdown("### Evidence-backed insights for capstone review"))
display(pd.DataFrame(evidence_backed_insights)[["id", "finding", "evidence", "action"]])
display(Markdown("### Presentation outline"))
display(pd.DataFrame(presentation_outline)[["slide", "title", "key_points"]])

print(f"Saved aggregate metrics: {METRICS_PATH}")
print(f"Saved report-safe figure: {FIGURE_PATH}")
print(json.dumps({
    "train_rows": metrics_payload["train_rows"],
    "test_rows": metrics_payload["test_rows"],
    "train_clients": metrics_payload["train_clients"],
    "test_clients": metrics_payload["test_clients"],
    "overlapping_clients": metrics_payload["overlapping_clients"],
    "model_precision_at_50": precision_rows[2]["Logistic Regression"],
    "leakage_precision_at_50": leakage_precision50,
    "leakage_roc_auc": leakage_roc_auc,
    "baseline_precision_at_50": precision_rows[2]["Week-4 baseline"],
    "test_label_rate": test_base_rate,
}, indent=2))
assert metrics_payload["overlapping_clients"] == 0
assert METRICS_PATH.exists() and FIGURE_PATH.exists()


### Evidence-backed insights for capstone review

,id,finding,evidence,action
0,I1,The learned ranking beats the transparent rule...,Precision@10: model 0.500 vs rule 0.100; Preci...,Keep the model as a candidate ordering aid for...
1,I2,"The strongest claim is top-K triage, not stron...",Precision@50 0.620 vs rule 0.320; test proxy r...,Choose K from actual reviewer capacity and val...
2,I3,Client grouping avoids client leakage but evid...,"Train: 23,837 rows / 25 clients; test: 6,163 r...",Repeat evaluation across multiple client-group...
3,I4,A label-source field can create deceptively ex...,Adding invalid trend_pct produces Precision@50...,Retain the feature-exclusion assertion and rer...


### Presentation outline

,slide,title,key_points
0,1,Problem and decision,Too many pages for manual review; rank the fir...
1,2,Data and target,"30,000-row anonymized starter snapshot; target..."
2,3,Method and validation,Logistic Regression vs transparent rule; group...
3,4,Results at reviewer capacity,"Precision@10/20/50/100; compare model, baselin..."
4,5,Leakage and limitations,trend_pct negative control inflates results; m...
5,6,Action playbook and next experiment,"Human-check measurement, intent, CTR, visibili..."


Saved aggregate metrics: work/outputs/capstone_metrics.json
Saved report-safe figure: work/figures/capstone_precision_at_k.png
{
  "train_rows": 23837,
  "test_rows": 6163,
  "train_clients": 25,
  "test_clients": 7,
  "overlapping_clients": 0,
  "model_precision_at_50": 0.62,
  "leakage_precision_at_50": 1.0,
  "leakage_roc_auc": 0.9551016606529538,
  "baseline_precision_at_50": 0.32,
  "test_label_rate": 0.5109524582184002
}


## 3. Evidence-backed insights and action implications

The preceding execution cell calculates and displays four findings from the current holdout rather than treating old hard-coded values as authoritative. Each finding includes the evidence and the decision it supports. The serialized versions are also stored in `work/outputs/capstone_metrics.json` for reviewer and CI checks.

The action playbook should remain human-reviewed. A high score means “inspect this record earlier,” not “make this edit” or “traffic will recover.” Use the reason-code checklist in the published report to verify measurement, indexing, intent, demand, and factual freshness before drafting an action.

## 4. Presentation outline

1. **Problem and decision:** Explain the page-review bottleneck and the narrow goal of prioritizing review, not guaranteeing recovery.
2. **Data and target:** Describe the 30,000-row anonymized starter snapshot, the observed `trend_direction == "down"` proxy, and the public-safe reporting boundary.
3. **Method and validation:** Compare Logistic Regression with the transparent rule baseline; show the client-grouped 80/20 holdout, seed 42, and zero client overlap.
4. **Results at reviewer capacity:** Present Precision@10, @20, @50, and @100 beside the baseline and test proxy rate; make Precision@50 the primary metric.
5. **Leakage and limitations:** Show why `trend_pct` is invalid, note modest ROC-AUC and only seven held-out clients, and explain why a snapshot proxy is not a future outcome.
6. **Action playbook and next experiment:** Walk through the human review checks, propose a future-window evaluation, and state the no-automatic-edit rule.



## 5. Reproducibility and self-check

- [x] Data download uses the public FlyRank starter dataset; no CSV is committed.
- [x] Final features exclude identifiers and label-derived fields; the separate `trend_pct` leakage control is marked invalid.
- [x] Train/test split is grouped by client and uses seed 42.
- [x] Metrics and a report-safe Precision@K chart are generated from the current run.
- [x] No raw records or row-level ranked queues are exported to Git.

Run all cells from a fresh runtime before submission. The workflow checks that the aggregate metrics are populated, the held-out client sets do not overlap, and the figure is valid.